## Datos a limpiar.

Gracias a la exploración inicial de los datos hemos podido encontrar una serie de datos que estropean la calidad de los datos. En esta sección vamos a limpiar y preparar los datos para su análisis. Por lo tanto, vamos a tratar los siguientes casos:

- Valores duplicados presentes.

- Identificadores de factura a nuestro modo de ver incorrectos.

- Identificadores de producto a nuestro modo de ver incorrectos.

- Descripciones de productos incorrectas y nulas.

- Descripciones duplicadas.

- Valores negativos y ceros en la columna 'Quantity'.

- Valores negativos y ceros en la columna 'UnitPrice'.

- Valores nulos en la columna 'CustomerID'.

- Tratamiento de outliers.


#### Librerías utilizadas.

In [300]:
import pandas as pd
import matplotlib.pyplot as plt

#### Carga del csv.

In [301]:
df = pd.read_csv('../data/data.csv', encoding='ISO-8859-1')

#### Valores duplicados.

Como hemos visto en la sección de exploración existen filas con valores duplicados que suman pesos en los análisis de manera incorrecta. Por lo tanto, vamos a eliminar estas filas para mejorar la calidad de los datos.

In [302]:
filas_pre_eliminar_duplicados = len(df)
df.drop_duplicates(inplace=True)
filas_tras_eliminar_duplicados = len(df)
print(f"Filas totales al inicio: {filas_pre_eliminar_duplicados}")
print(f"Filas totales tras eliminar duplicados: {filas_tras_eliminar_duplicados}")

Filas totales al inicio: 541909
Filas totales tras eliminar duplicados: 536641


#### Identificadores de factura.

Como vimos, los 'InvoiceNo' que tienen 7 caracteres siempre empiezan por la letra 'C' o por la letra 'A', lo que indica que son facturas canceladas o cobros mal realizados. Estas filas no aportan información útil para el análisis, por lo que vamos a eliminarlas.  

In [303]:
filas_pre_eliminar_invoices = len(df)
df = df[df['InvoiceNo'].notna() & (df['InvoiceNo'].str.len() == 6)]
filas_tras_eliminar_invoices = len(df)
print(f"Filas totales tras eliminar InvoiceNo inválidos: {filas_tras_eliminar_invoices} (eliminadas {filas_pre_eliminar_invoices - filas_tras_eliminar_invoices})")

Filas totales tras eliminar InvoiceNo inválidos: 527387 (eliminadas 9254)


#### Identificadores de producto.

Detectamos que la gran mayoría de StockCode (99,4%) se comprenden entre 5 y 6 caracteres de longitud y que el resto tenían algunas descripciones que no correspondían a una venta normal. Por lo tanto, vamos a eliminar los StockCode que no tengan 5 o 6 caracteres para evitar complicaciones en el análisis.

In [304]:
filas_pre_eliminar_stockcode = len(df)
df = df[df['StockCode'].notna() & (df['StockCode'].str.len().between(5, 6))]
filas_tras_eliminar_stockcode = len(df)
print(f"Filas totales tras eliminar StockCode inválidos: {filas_tras_eliminar_stockcode} (eliminadas {filas_pre_eliminar_stockcode - filas_tras_eliminar_stockcode})")

Filas totales tras eliminar StockCode inválidos: 524599 (eliminadas 2788)


#### Descripciones inválidas.

Las descripciones de los productos nos aportan información para conocer si se produjo un venta normal o si se trató de un error. Detectamos que las descripciones en minúscula no representan información sobre el producto, sino una descripción de un error o mala gestión. Por lo tanto procedemos a eliminar estas filas para asegurar una mejor calidad de los datos.

In [305]:
filas_pre_eliminar_descripciones = len(df)
df = df[df['Description'].notna() & (df['Description'].str.isupper())]
filas_tras_eliminar_descripciones = len(df)
print(f"Filas totales tras eliminar descripciones inválidas: {filas_tras_eliminar_descripciones} (eliminadas {filas_pre_eliminar_descripciones - filas_tras_eliminar_descripciones})")

Filas totales tras eliminar descripciones inválidas: 520843 (eliminadas 3756)


#### Descripciones duplicadas.

En la sección de exploración vimos que también existían descripciones en mayúscula (válidas) similares para un mismo producto. Realmente no es un error, son casos en los que la descripción varía ligeramente entre unos casos y otros, pero hacen referencia al mismo producto. Por lo que no es necesario eliminar estas filas, sólo vamos a unificar la descripción para cada StockCode con la más frecuente.

In [306]:
result = df.groupby('StockCode')['Description'].apply(lambda x: x.unique())
multi_desc = result[result.apply(len) > 1]
print(f"Filas totales pre unificar descripciones: {len(df)}")
print(f"StockCodes con múltiples descripciones pre unificar: {len(multi_desc)}")

df['Description'] = df.groupby('StockCode')['Description'].transform(lambda x: x.value_counts().index[0])

descripciones_por_stock = df.groupby('StockCode')['Description'].nunique()
multi_desc = descripciones_por_stock[descripciones_por_stock > 1]
print(f"Filas totales post unificar descripciones: {len(df)}")
print(f"StockCodes con múltiples descripciones post unificar: {len(multi_desc)}")

Filas totales pre unificar descripciones: 520843
StockCodes con múltiples descripciones pre unificar: 232
Filas totales post unificar descripciones: 520843
StockCodes con múltiples descripciones post unificar: 0


#### Valores negativos y ceros en la columna 'Quantity'.

Los valores negativos y ceros en la columnna 'Quantity' no representan ventas normales, sino devoluciones o errores. Por lo tanto, vamos a eliminar estas filas para mejorar la calidad de los datos. Vemos que ya no quedaban tantas tras las limpiezas anteriores, pero aún así es importante eliminarlas para no estropear el análisis.

In [307]:
negativos_ceros_quantity = df[df['Quantity'] <= 0]
print(f"Filas totales pre eliminar Quantity <= 0: {len(df)}")
print(f"Filas con Quantity <= 0: {len(negativos_ceros_quantity)}")

df = df[df['Quantity'] > 0]
print(f"Filas totales post eliminar Quantity <= 0: {len(df)}")

Filas totales pre eliminar Quantity <= 0: 520843
Filas con Quantity <= 0: 8
Filas totales post eliminar Quantity <= 0: 520835


#### Valores negativos y ceros en la columna 'UnitPrice'.

También es importante eliminar los valores negativos y ceros en la columna 'UnitPrice' ya que no representan ventas normales.

In [308]:
negativos_ceros_unitprice = df[df['UnitPrice'] <= 0]
print(f"Filas totales pre eliminar UnitPrice <= 0: {len(df)}")
print(f"Filas con UnitPrice <= 0: {len(negativos_ceros_unitprice)}")

df = df[df['UnitPrice'] > 0]
print(f"Filas totales post eliminar UnitPrice <= 0: {len(df)}")

Filas totales pre eliminar UnitPrice <= 0: 520835
Filas con UnitPrice <= 0: 396
Filas totales post eliminar UnitPrice <= 0: 520439


#### Nulos en identificador de cliente.

Los identificadores de cliente nulos son un tema delicado en este conjunto de datos, ya que representaban un porcentaje significativo de las filas en el análisis inicial (24,9%).

Vamos a comprobar cómo se encuentra el estado actual.

In [309]:
nulos_customerid = df[df['CustomerID'].isna()]
print(f"Filas con CustomerID nulos: {len(nulos_customerid)}")

Filas con CustomerID nulos: 130935


Seguimos teniendo un porcentaje significativo de filas con CustomerID nulos, por lo que vamos a intentar recuperar algunos valores. Parece que no hay ningún CustomerID válido en la factura cuando hay nulos. Es decir, las facturas son totalmente nulas en todas las operaciones. Por lo tanto tendremos que prescindir o ignorar estas filas dependiendo de los futuros objetivos. 

In [310]:
agrupados_por_invoice = df.groupby('InvoiceNo')['CustomerID'].agg(['count', 'nunique'])
print("Estadísticas de CustomerID por InvoiceNo:")
print(agrupados_por_invoice.head(10))

facturas_con_nulos_y_validos = df.groupby('InvoiceNo')['CustomerID'].apply(lambda x: x.isna().any() and x.notna().any())
facturas_mixtas = facturas_con_nulos_y_validos[facturas_con_nulos_y_validos]
print(f"\nFacturas con CustomerID nulos y válidos: {len(facturas_mixtas)}")

df['CustomerID'] = df.groupby('InvoiceNo')['CustomerID'].transform(lambda x: x.fillna(x.max()))

nulos_restantes = df[df['CustomerID'].isna()]
print(f"CustomerID nulos tras recuperación: {len(nulos_restantes)}")

Estadísticas de CustomerID por InvoiceNo:
           count  nunique
InvoiceNo                
536365         7        1
536366         2        1
536367        12        1
536368         4        1
536369         1        1
536370        19        1
536371         1        1
536372         2        1
536373        16        1
536374         1        1

Facturas con CustomerID nulos y válidos: 0
CustomerID nulos tras recuperación: 130935


#### Tratamiento de outliers.

Para finalizar la limpieza es necesario deshacerse de valores muy alejados de los valores típicos que pueden estropear el análisis. 

In [311]:
filas_pre_eliminar_outliers = len(df)

quantity_Q1 = df["Quantity"].quantile(0.20)
quantity_Q3 = df["Quantity"].quantile(0.80)
quantity_IQR = quantity_Q3 - quantity_Q1
limite_inferior_quantity = quantity_Q1 - 1.5 * quantity_IQR
limite_superior_quantity = quantity_Q3 + 1.5 * quantity_IQR

outliers_quantity = df[(df["Quantity"] < limite_inferior_quantity) | (df["Quantity"] > limite_superior_quantity)]
print(f"Outliers en Quantity: {len(outliers_quantity)}")
print(f"Límites: [{limite_inferior_quantity:.2f}, {limite_superior_quantity:.2f}]")

unit_price_Q1 = df["UnitPrice"].quantile(0.20)
unit_price_Q3 = df["UnitPrice"].quantile(0.80)
unit_price_IQR = unit_price_Q3 - unit_price_Q1
unit_price_lower = unit_price_Q1 - 1.5 * unit_price_IQR
unit_price_upper = unit_price_Q3 + 1.5 * unit_price_IQR

outliers_unitPrice = df[(df["UnitPrice"] < unit_price_lower) | (df["UnitPrice"] > unit_price_upper)]
print(f"\nOutliers en UnitPrice: {len(outliers_unitPrice)}")
print(f"Límites: [{unit_price_lower:.2f}, {unit_price_upper:.2f}]")

df = df[(df["Quantity"] >= limite_inferior_quantity) & (df["Quantity"] <= limite_superior_quantity)]
df = df[(df["UnitPrice"] >= unit_price_lower) & (df["UnitPrice"] <= unit_price_upper)]

filas_tras_eliminar_outliers = len(df)
print(f"\nFilas totales pre eliminar outliers: {filas_pre_eliminar_outliers}")
print(f"Filas totales post eliminar outliers: {filas_tras_eliminar_outliers} (eliminadas {filas_pre_eliminar_outliers - filas_tras_eliminar_outliers})")


Outliers en Quantity: 26686
Límites: [-15.50, 28.50]

Outliers en UnitPrice: 16920
Límites: [-5.30, 11.10]

Filas totales pre eliminar outliers: 520439
Filas totales post eliminar outliers: 476888 (eliminadas 43551)


#### Resumen de la limpieza.

In [312]:
print(f"\nFilas finales tras limpieza: {len(df)}")
print(f"\nColumnas del dataset: {list(df.columns)}")
print(f"\nTipo de datos:\n{df.dtypes}\n\n")

print("Valores nulos\n")
print(df.isnull().sum())

print(f"\n\nFacturas únicas: {df['InvoiceNo'].nunique()}")
print(f"Clientes únicos: {df['CustomerID'].nunique()}")
print(f"Productos únicos: {df['StockCode'].nunique()}")
print(f"Países únicos: {df['Country'].nunique()}\n")

print(f"\nCantidad (Quantity):")
print(f"  Mínimo: {df['Quantity'].min()}")
print(f"  Máximo: {df['Quantity'].max()}")
print(f"  Media: {df['Quantity'].mean():.2f}")

print(f"\nPrecio unitario (UnitPrice):")
print(f"  Mínimo: {df['UnitPrice'].min():.2f}")
print(f"  Máximo: {df['UnitPrice'].max():.2f}")
print(f"  Media: {df['UnitPrice'].mean():.2f}")

print(f"\nRango de fechas:")
print(f"  Desde: {df['InvoiceDate'].min()}")
print(f"  Hasta: {df['InvoiceDate'].max()}\n\n")


Filas finales tras limpieza: 476888

Columnas del dataset: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Tipo de datos:
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object


Valores nulos

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     120892
Country             0
dtype: int64


Facturas únicas: 18338
Clientes únicos: 4208
Productos únicos: 3774
Países únicos: 38


Cantidad (Quantity):
  Mínimo: 1
  Máximo: 28
  Media: 6.06

Precio unitario (UnitPrice):
  Mínimo: 0.06
  Máximo: 11.05
  Media: 2.87

Rango de fechas:
  Desde: 1/10/2011 10:32
  Hasta: 9/9/2011 9:52




#### Dataset limpio para regresión.

Como tenemos valores nulos en el cliente que no hemos podido resolver, vamos a mantenerlo para el modelo de regresión ya que no es importante para el análisis de ventas diarias.

In [313]:
import os
os.makedirs('../output', exist_ok=True)

df.to_csv('../output/data_limpio_regresion.csv', index=False, encoding='ISO-8859-1')
print(f"Dataset de regresión exportado: {len(df)} filas")
print("Archivo: ../output/data_limpio_regresion.csv")

Dataset de regresión exportado: 476888 filas
Archivo: ../output/data_limpio_regresion.csv


#### Dataset limpio para clustering.

En el caso del clustering, al necesitar la variable de clientes para segmentarlos, no podemos contar con valores nulos, por lo que los eliminamos para este caso.

In [314]:
import os
os.makedirs('../output', exist_ok=True)

df_clustering = df[df['CustomerID'].notna()].copy()
filas_eliminadas_clustering = len(df) - len(df_clustering)

df_clustering.to_csv('../output/data_limpio_clustering.csv', index=False, encoding='ISO-8859-1')
print(f"Dataset de clustering exportado: {len(df_clustering)} filas")
print(f"Filas eliminadas por CustomerID nulo: {filas_eliminadas_clustering}")
print("Archivo: ../output/data_limpio_clustering.csv")

Dataset de clustering exportado: 355996 filas
Filas eliminadas por CustomerID nulo: 120892
Archivo: ../output/data_limpio_clustering.csv
